In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Helper to process metric/trace/log files
def process_timeseries_file(filename, kpi_col, source_name):
    df = pd.read_csv(filename)
    # Normalize column name for KPI
    df = df.rename(columns={kpi_col: 'kpi_name'})
    # Ensure expected columns
    expected = {'timestamp', 'cmdb_id', 'kpi_name', 'value'}
    if not expected.issubset(df.columns):
        raise ValueError(f"{filename} missing columns. Expected at least: {expected}")
    # Convert timestamp to int and keep original for aggregation then convert to tz-aware datetime for display
    df['timestamp'] = df['timestamp'].astype(int)
    # Compute uniques (prepare short/sample strings)
    unique_cmdb = df['cmdb_id'].dropna().unique()
    unique_kpis = df['kpi_name'].dropna().unique()
    unique_cmdb_sample = ", ".join(map(str, unique_cmdb[:20]))  # short list sample
    unique_kpi_sample = ", ".join(map(str, unique_kpis[:20]))
    unique_summary = {
        'source': source_name,
        'unique_cmdb_count': len(unique_cmdb),
        'unique_cmdb_sample': unique_cmdb_sample,
        'unique_kpi_count': len(unique_kpis),
        'unique_kpi_sample': unique_kpi_sample
    }
    # Aggregate per (cmdb_id, kpi_name)
    agg = df.groupby(['cmdb_id', 'kpi_name']).agg(
        p95 = ('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
        count = ('value', 'count'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    ).reset_index()
    # Convert min/max timestamps to timezone-aware ISO strings in Asia/Shanghai
    agg['min_timestamp'] = pd.to_datetime(agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    agg['max_timestamp'] = pd.to_datetime(agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # Format timestamps compactly
    agg['min_timestamp'] = agg['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['max_timestamp'] = agg['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg.insert(0, 'source', source_name)
    # Keep columns in requested order
    agg = agg[['source', 'cmdb_id', 'kpi_name', 'p95', 'count', 'min_timestamp', 'max_timestamp']]
    return df, pd.DataFrame([unique_summary]), agg

# Process metric.csv
metric_df, metric_uniques_df, metric_agg = process_timeseries_file('metric.csv', 'kpi_name', 'metric')

# Process trace.csv
trace_df, trace_uniques_df, trace_agg = process_timeseries_file('trace.csv', 'trace_name', 'trace')

# Process log.csv
log_df, log_uniques_df, log_agg = process_timeseries_file('log.csv', 'log_name', 'log')

# Combine uniques summaries into one compact table
uniques_summary = pd.concat([metric_uniques_df, trace_uniques_df, log_uniques_df], ignore_index=True)

# Combine aggregated results and limit to top 50 rows for compactness (sort for determinism)
combined_agg = pd.concat([metric_agg, trace_agg, log_agg], ignore_index=True)
combined_agg = combined_agg.sort_values(['source', 'cmdb_id', 'kpi_name']).reset_index(drop=True)
combined_agg_top50 = combined_agg.head(50)

# Process error_logs.csv: unique cmdb_id list (short) and counts per cmdb_id (top 20)
err = pd.read_csv('error_logs.csv')
err['timestamp'] = err['timestamp'].astype(int)
err_unique_cmdb = err['cmdb_id'].dropna().unique()
err_unique_sample = ", ".join(map(str, err_unique_cmdb[:20]))
err_uniques_summary = pd.DataFrame([{
    'file': 'error_logs',
    'unique_cmdb_count': len(err_unique_cmdb),
    'unique_cmdb_sample': err_unique_sample
}])
err_counts = err.groupby('cmdb_id').size().rename('count').reset_index().sort_values('count', ascending=False)
err_counts_top20 = err_counts.head(20)

# Display compact outputs (variables)
uniques_summary, combined_agg_top50, err_uniques_summary, err_counts_top20

```
Out[1]:
```
```python
# Summarize the execution results in plain English using existing variables from the session.
# The final displayed output is a single compact summary string variable.

# Reuse existing variables: uniques_summary, combined_agg_top50, err_uniques_summary, err_counts_top20

metric_row = uniques_summary.loc[uniques_summary['source']=='metric'].iloc[0]
trace_row = uniques_summary.loc[uniques_summary['source']=='trace'].iloc[0]
log_row = uniques_summary.loc[uniques_summary['source']=='log'].iloc[0]

# Build short samples (they are already short sampled strings in those rows)
metric_sample = f"{metric_row['unique_cmdb_count']} cmdb_ids (sample: {metric_row['unique_cmdb_sample'][:200]}...), {metric_row['unique_kpi_count']} unique KPIs (sample: {metric_row['unique_kpi_sample'][:200]}...)"
trace_sample = f"{trace_row['unique_cmdb_count']} cmdb_ids (sample: {trace_row['unique_cmdb_sample'][:200]}...), {trace_row['unique_kpi_count']} unique trace names (sample: {trace_row['unique_kpi_sample'][:200]}...)"
log_sample = f"{log_row['unique_cmdb_count']} cmdb_ids (sample: {log_row['unique_cmdb_sample'][:200]}...), {log_row['unique_kpi_count']} unique log names (sample: {log_row['unique_kpi_sample'][:200]}...)"

# Example aggregated row from combined_agg_top50 (choose first row for a concrete example)
example_row = combined_agg_top50.iloc[0]
example_desc = (f"Example aggregated entry: source={example_row['source']}, cmdb_id={example_row['cmdb_id']}, "
                f"kpi_name={example_row['kpi_name']}, p95={example_row['p95']:.2f}, count={int(example_row['count'])}, "
                f"min_timestamp={example_row['min_timestamp']}, max_timestamp={example_row['max_timestamp']}")

# Error logs summary: unique count and top counts
err_uniques = err_uniques_summary.iloc[0]
err_top = err_counts_top20.copy()
err_top_list = ", ".join([f"{r['cmdb_id']}:{int(r['count'])}" for _, r in err_top.iterrows()[:3]])

summary = (
    "Summary of telemetry files (compact):\n\n"
    f"Metric file: {metric_sample}\n"
    f"Trace file:  {trace_sample}\n"
    f"Log file:    {log_sample}\n\n"
    "Aggregations computed: For each (cmdb_id, KPI/trace/log name) we calculated global P95 of 'value', total count, "
    "min timestamp and max timestamp. Combined aggregation table (top 50 rows shown) contains these rows.\n\n"
    f"{example_desc}\n\n"
    "Error logs: "
    f"{int(err_uniques['unique_cmdb_count'])} unique cmdb_ids (sample: {err_uniques['unique_cmdb_sample']}). "
    "Top error log counts (top 3): " + err_top_list + ".\n\n"
    "If you want more detail (e.g., specific aggregated rows or more examples), ask for df.head(N) on the combined aggregate."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   source  unique_cmdb_count                                 unique_cmdb_sample  unique_kpi_count                                  unique_kpi_sample
0  metric                 59  adservice, adservice-0, adservice-1, adservice...              1176  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
1   trace                 40  adservice-0, adservice-1, adservice-2, adservi...               200  trace.from_frontend-0.duration_mean, trace.fro...
2     log                 31  adservice-0, adservice-1, adservice-2, cartser...                 2                     log.error_count, log.row_count,    source                  cmdb_id         kpi_name     p95  count            min_timestamp            max_timestamp
0     log              adservice-0  log.error_count    0.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
1     log              adservice-0    log.row_count  594.30     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
2     log              adservice-1  log.error_count    0.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
3     log              adservice-1    log.row_count   51.10     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
4     log              adservice-2  log.error_count    0.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
..    ...                      ...              ...     ...    ...                      ...                      ...
45    log  productcatalogservice-1    log.row_count  463.95     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
46    log  productcatalogservice-2  log.error_count    0.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
47    log  productcatalogservice-2    log.row_count  461.75     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
48    log  recommendationservice-0  log.error_count    0.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST
49    log  recommendationservice-0    log.row_count  102.00     30  2022-03-20 22:00:00 CST  2022-03-20 22:29:00 CST

[50 rows x 7 columns],          file  unique_cmdb_count                  unique_cmdb_sample
0  error_logs                  3  frontend-2, frontend-0, frontend-1,       cmdb_id  count
2  frontend-2    457
0  frontend-0    149
1  frontend-1     77)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# --- Load raw CSVs ---
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
err_df = pd.read_csv('error_logs.csv')

# Normalize columns: unify KPI column name to 'kpi_name' and ensure 'value' numeric and 'timestamp' int
metric_df = metric_df.rename(columns={'kpi_name': 'kpi_name'})
trace_df = trace_df.rename(columns={'trace_name': 'kpi_name'})
log_df = log_df.rename(columns={'log_name': 'kpi_name'})

for df in (metric_df, trace_df, log_df):
    df['timestamp'] = df['timestamp'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

err_df['timestamp'] = err_df['timestamp'].astype(int)

# --- 1) Compute global P95 thresholds per (source, cmdb_id, kpi_name) using full series ---
def compute_thresholds(df, source_label):
    thr = df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index()
    thr = thr.rename(columns={'value': 'p95'})
    thr.insert(0, 'source', source_label)
    return thr

metric_thr = compute_thresholds(metric_df, 'metric')
trace_thr = compute_thresholds(trace_df, 'trace')
log_thr = compute_thresholds(log_df, 'log')

thresholds = pd.concat([metric_thr, trace_thr, log_thr], ignore_index=True)
# Keep compact and stable sorting
thresholds = thresholds.sort_values(['source', 'cmdb_id', 'kpi_name']).reset_index(drop=True)

# --- 2) Incident window filter: '2022-03-20 22:00:00 CST' to '2022-03-20 22:29:00 CST' inclusive ---
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
# convert to epoch seconds (UTC-based)
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

metric_win = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
trace_win  = trace_df[(trace_df['timestamp']  >= start_ts) & (trace_df['timestamp']  <= end_ts)].copy()
log_win    = log_df[(log_df['timestamp']    >= start_ts) & (log_df['timestamp']    <= end_ts)].copy()

# Add source column for joining
metric_win['source'] = 'metric'
trace_win['source']  = 'trace'
log_win['source']    = 'log'

# --- 3) Join filtered rows with thresholds and compute anomaly metrics ---
def analyze_window(win_df, thresholds_df):
    # Merge to bring p95 per row
    merged = win_df.merge(thresholds_df, on=['source', 'cmdb_id', 'kpi_name'], how='inner')
    # Anomaly flag where value > p95 (strictly greater as requested)
    merged['is_anomaly'] = merged['value'] > merged['p95']
    # Group and aggregate required metrics
    def earliest_anom_ts(series_ts, series_flag):
        if series_flag.any():
            t = series_ts[series_flag].min()
            return pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            return pd.NA

    agg = merged.groupby(['source', 'cmdb_id', 'kpi_name'], dropna=False).agg(
        p95 = ('p95', 'first'),
        total_points_in_window = ('value', 'count'),
        anomaly_count = ('is_anomaly', 'sum'),
        max_value_in_window = ('value', 'max'),
        mean_value_in_window = ('value', 'mean'),
        _min_ts = ('timestamp', 'min')  # keep for potential use
    ).reset_index()

    # Compute earliest_anomaly_timestamp via groupby apply to avoid losing row-level filter
    ea = merged[merged['is_anomaly']].groupby(['source','cmdb_id','kpi_name'], dropna=False)['timestamp'].min().reset_index()
    if not ea.empty:
        ea['earliest_anomaly_timestamp'] = pd.to_datetime(ea['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        ea = ea.drop(columns=['timestamp'])
        agg = agg.merge(ea, on=['source','cmdb_id','kpi_name'], how='left')
    else:
        agg['earliest_anomaly_timestamp'] = pd.NA

    # Format numeric columns compactly and timestamps
    agg['p95'] = agg['p95'].astype(float)
    agg['max_value_in_window'] = agg['max_value_in_window'].astype(float)
    agg['mean_value_in_window'] = agg['mean_value_in_window'].astype(float)
    agg['mean_value_in_window'] = agg['mean_value_in_window'].round(5)
    agg['max_value_in_window'] = agg['max_value_in_window'].round(5)
    agg['p95'] = agg['p95'].round(5)

    # Ensure earliest_anomaly_timestamp present (NA if no anomalies)
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].fillna(pd.NA)

    # Keep only required columns
    agg = agg[['source','cmdb_id','kpi_name','p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window']]
    return agg

an_metric = analyze_window(metric_win, thresholds[thresholds['source']=='metric'])
an_trace  = analyze_window(trace_win,  thresholds[thresholds['source']=='trace'])
an_log    = analyze_window(log_win,    thresholds[thresholds['source']=='log'])

anomalies = pd.concat([an_metric, an_trace, an_log], ignore_index=True)
# Replace possible NaN anomaly_count dtype to int
anomalies['anomaly_count'] = anomalies['anomaly_count'].astype(int)
anomalies['total_points_in_window'] = anomalies['total_points_in_window'].astype(int)

# --- 4) Top 20 sorted by anomaly_count desc, then max_value_in_window desc ---
anomaly_candidates = anomalies.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# --- 5) Error logs in same incident window: counts per cmdb_id, min and max timestamps in window ---
err_win = err_df[(err_df['timestamp'] >= start_ts) & (err_df['timestamp'] <= end_ts)].copy()
err_summary = err_win.groupby('cmdb_id', dropna=False).agg(
    count = ('timestamp','count'),
    min_timestamp = ('timestamp','min'),
    max_timestamp = ('timestamp','max')
).reset_index()

# format timestamps to Asia/Shanghai
err_summary['min_timestamp'] = pd.to_datetime(err_summary['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
err_summary['max_timestamp'] = pd.to_datetime(err_summary['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

err_summary_top20 = err_summary.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs: anomaly_candidates (top 20) and err_summary_top20
anomaly_candidates, err_summary_top20

```
Out[2]:
```
```python
# Summarize the anomaly and error-log results from the previous analysis as a compact plain-English string.
top_anoms = anomaly_candidates.head(5)
top_errs = err_summary_top20.head(3)

lines = []
lines.append("Incident window: 2022-03-20 22:00:00 CST to 2022-03-20 22:29:00 CST.")
lines.append("")
lines.append("Top anomaly candidates (source, cmdb_id, kpi_name, p95, anomaly_count, max_value_in_window):")
for r in top_anoms.itertuples(index=False):
    lines.append(f"- {r.source}, {r.cmdb_id}, {r.kpi_name}, p95={r.p95:.5g}, anomalies={int(r.anomaly_count)}, max={r.max_value_in_window:.5g}")

lines.append("")
lines.append("Error logs in the same window (top 3 by count):")
for r in top_errs.itertuples(index=False):
    lines.append(f"- {r.cmdb_id}: count={int(r.count)}, earliest={r.min_timestamp}, latest={r.max_timestamp}")

lines.append("")
lines.append("Notes:")
lines.append("- All listed anomaly candidates have 30 data points in the window (full 30-minute window) and small anomaly counts (mostly 2).")
lines.append("- Metric anomalies are concentrated on adservice / adservice2 and several node-level FS/disk metrics (node-1..node-6).")
lines.append("- Error log volume is highest on frontend pods (frontend-2 >> frontend-0 > frontend-1).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source     cmdb_id                                           kpi_name           p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0   metric   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  8.590390e+11                      30              2    2022-03-20 22:28:00 CST         8.592683e+11          8.569392e+11
1   metric  adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  9.564433e+10                      30              2    2022-03-20 22:28:00 CST         9.589333e+10          9.345386e+10
2   metric   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  8.107171e+10                      30              2    2022-03-20 22:27:00 CST         8.111755e+10          8.083007e+10
3   metric   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  7.886088e+10                      30              2    2022-03-20 22:27:00 CST         7.891417e+10          7.862444e+10
4   metric   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  6.422520e+10                      30              2    2022-03-20 22:28:00 CST         6.424679e+10          6.403416e+10
5   metric      node-4                             system.fs.inodes.total  6.612278e+09                      30              2    2022-03-20 22:00:00 CST         6.613120e+09          6.608281e+09
6   metric      node-4                              system.fs.inodes.free  6.609691e+09                      30              2    2022-03-20 22:00:00 CST         6.610534e+09          6.605693e+09
7   metric  adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  6.444928e+09                      30              2    2022-03-20 22:28:00 CST         6.503962e+09          6.190398e+09
8   metric  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  6.007625e+09                      30              2    2022-03-20 22:28:00 CST         6.062500e+09          5.773389e+09
9   metric      node-4                                   system.disk.used  5.768349e+09                      30              2    2022-03-20 22:25:00 CST         5.768376e+09          5.767848e+09
10  metric      node-6                             system.fs.inodes.total  4.968760e+09                      30              2    2022-03-20 22:00:00 CST         4.969808e+09          4.336541e+09
11  metric      node-1                             system.fs.inodes.total  4.969030e+09                      30              2    2022-03-20 22:00:00 CST         4.969574e+09          4.966022e+09
12  metric      node-2                             system.fs.inodes.total  4.969030e+09                      30              2    2022-03-20 22:00:00 CST         4.969565e+09          4.966015e+09
13  metric      node-3                             system.fs.inodes.total  4.968752e+09                      30              2    2022-03-20 22:00:00 CST         4.969514e+09          4.965951e+09
14  metric      node-6                              system.fs.inodes.free  4.966671e+09                      30              2    2022-03-20 22:00:00 CST         4.967720e+09          4.334658e+09
15  metric      node-1                              system.fs.inodes.free  4.967091e+09                      30              2    2022-03-20 22:00:00 CST         4.967635e+09          4.964082e+09
16  metric      node-2                              system.fs.inodes.free  4.967061e+09                      30              2    2022-03-20 22:00:00 CST         4.967596e+09          4.964045e+09
17  metric      node-3                              system.fs.inodes.free  4.966805e+09                      30              2    2022-03-20 22:00:00 CST         4.967566e+09          4.964004e+09
18  metric  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  4.483383e+09                      30              2    2022-03-20 22:28:00 CST         4.506072e+09          4.291788e+09
19  metric      node-6                                   system.disk.used  4.388252e+09                      30              2    2022-03-20 22:00:00 CST         4.389392e+09          3.839191e+09,       cmdb_id  count            min_timestamp            max_timestamp
0  frontend-2    443  2022-03-20 22:00:01 CST  2022-03-20 22:28:56 CST
1  frontend-0    144  2022-03-20 22:00:07 CST  2022-03-20 22:28:46 CST
2  frontend-1     73  2022-03-20 22:00:16 CST  2022-03-20 22:28:38 CST)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables from kernel:
# anomaly_candidates, metric_win, trace_win, log_win, thresholds

# 1) Select top 20 by anomaly_count from anomaly_candidates
top20_pairs = anomaly_candidates.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# 2) Combine filtered incident-window raw rows
window_df = pd.concat([metric_win, trace_win, log_win], ignore_index=True)
# Ensure kpi_name column present and types
window_df['kpi_name'] = window_df['kpi_name'].astype(str)
window_df['cmdb_id'] = window_df['cmdb_id'].astype(str)
window_df['source'] = window_df['source'].astype(str)
window_df['timestamp'] = window_df['timestamp'].astype(int)
window_df['value'] = pd.to_numeric(window_df['value'], errors='coerce')

# Merge with thresholds to get p95 per row
thr = thresholds.copy()
thr['kpi_name'] = thr['kpi_name'].astype(str)
merged_win = window_df.merge(thr, on=['source','cmdb_id','kpi_name'], how='left')

# Prepare results list
results = []

def format_ts(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

for r in top20_pairs.itertuples(index=False):
    src, cmdb, kpi, p95_val, total_pts, anom_cnt = r.source, r.cmdb_id, r.kpi_name, float(r.p95), int(r.total_points_in_window), int(r.anomaly_count)
    # Filter merged rows for this pair
    pair_rows = merged_win[(merged_win['source']==src) & (merged_win['cmdb_id']==cmdb) & (merged_win['kpi_name']==kpi)].copy()
    # If no p95 available in merged (NaN) try to lookup from thresholds
    if pair_rows['p95'].isna().all():
        p95_val = float(thr[(thr['source']==src)&(thr['cmdb_id']==cmdb)&(thr['kpi_name']==kpi)]['p95'].iloc[0]) if not thr[(thr['source']==src)&(thr['cmdb_id']==cmdb)&(thr['kpi_name']==kpi)].empty else float('nan')
        pair_rows['p95'] = p95_val
    # Identify anomaly rows (value > p95). Handle possible NaNs.
    pair_rows = pair_rows.dropna(subset=['value','p95'])
    anom_rows = pair_rows[pair_rows['value'] > pair_rows['p95']].copy()
    anom_rows = anom_rows.sort_values('timestamp').reset_index(drop=True)
    anomaly_count = int(len(anom_rows))
    if anomaly_count == 0:
        longest_run = 0
        earliest_run_start = pd.NA
        run_max_value = pd.NA
        severity_ratio = pd.NA
    else:
        ts_arr = anom_rows['timestamp'].to_numpy(dtype=int)
        vals = anom_rows['value'].to_numpy(dtype=float)
        # Find longest consecutive run where diff == 60
        max_run = 1
        max_run_start_idx = 0
        cur_run = 1
        cur_start = 0
        for i in range(1, len(ts_arr)):
            if ts_arr[i] - ts_arr[i-1] == 60:
                cur_run += 1
            else:
                if cur_run > max_run:
                    max_run = cur_run
                    max_run_start_idx = cur_start
                cur_run = 1
                cur_start = i
        # Final check
        if cur_run > max_run:
            max_run = cur_run
            max_run_start_idx = cur_start
        longest_run = int(max_run)
        # earliest_run_start_timestamp: timestamp at start idx
        earliest_run_start = format_ts(int(ts_arr[max_run_start_idx]))
        # run_max_value: max value in that run
        run_vals = vals[max_run_start_idx:max_run_start_idx+max_run]
        run_max_value = float(np.max(run_vals))
        # severity_ratio = (run_max_value - p95) / p95
        # Use p95_val (global p95) from thresholds (should be same as pair_rows['p95'].iloc[0])
        p95_used = float(pair_rows['p95'].iloc[0]) if not pair_rows['p95'].isna().all() else float('nan')
        if p95_used == 0 or np.isnan(p95_used):
            severity_ratio = float('nan')
        else:
            severity_ratio = float((run_max_value - p95_used) / p95_used)
    results.append({
        'source': src,
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95': float(p95_val),
        'total_points_in_window': int(total_pts),
        'anomaly_count': int(anomaly_count),
        'longest_consecutive_run_minutes': int(longest_run),
        'earliest_run_start_timestamp': earliest_run_start,
        'run_max_value': run_max_value,
        'severity_ratio': severity_ratio
    })

res_df = pd.DataFrame(results)

# 4) Sort by longest_consecutive_run_minutes desc, then severity_ratio desc, then anomaly_count desc
res_df_sorted = res_df.sort_values(['longest_consecutive_run_minutes','severity_ratio','anomaly_count'], ascending=[False, False, False]).reset_index(drop=True)
final_table = res_df_sorted.head(20)

# 5) Check if any have longest_consecutive_run_minutes >= 3
if (final_table['longest_consecutive_run_minutes'] >= 3).any():
    note = pd.NA
else:
    note = "No multi-minute consecutive faults (>=3 minutes) found among top 20 candidates."

# Keep outputs compact
final_table[['source','cmdb_id','kpi_name','p95','total_points_in_window','anomaly_count',
             'longest_consecutive_run_minutes','earliest_run_start_timestamp','run_max_value','severity_ratio']], note

```
Out[3]:
```
```python
# Summarize the findings from the previously computed table as a compact plain-English string.
summary = (
    "Incident window analyzed: 2022-03-20 22:00:00 CST to 2022-03-20 22:29:00 CST.\n\n"
    "Key findings (top 20 anomaly candidates):\n"
    "- All top candidates are metric-series. Each pair has 30 points in the window (full 30 minutes).\n"
    "- Anomaly counts are small (mostly 2 anomalies per (source,cmdb_id,kpi)).\n"
    "- Longest consecutive anomaly runs are short: many candidates show a 2-minute consecutive run (common for adservice2 and several node fs/disk metrics),\n"
    "  some show only 1-minute runs. Example run start times: adservice2 runs start at 2022-03-20 22:28:00 CST; adservice runs at ~22:27:00 CST; several node fs metrics at ~22:00:00 CST; node-4 disk at ~22:25:00 CST.\n"
    "- Severity ratios are very small (examples ~0.009 down to ~0.000002), meaning the peak values exceed the global P95 by only a tiny relative amount.\n\n"
    "Conclusion:\n"
    "- No sustained multi-minute consecutive faults were found among the top 20 candidates (no runs >= 3 minutes).\n"
    "- Observed anomalies are short (1–2 minute bursts) and low-severity relative to the global P95 thresholds; these look like brief, marginal exceedances rather than prolonged/systemic failures."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source     cmdb_id                                           kpi_name           p95  total_points_in_window  anomaly_count  longest_consecutive_run_minutes earliest_run_start_timestamp  run_max_value  severity_ratio
0   metric  adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  6.444928e+09                      30              2                                2      2022-03-20 22:28:00 CST   6.503962e+09        0.009160
1   metric  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  6.007625e+09                      30              2                                2      2022-03-20 22:28:00 CST   6.062500e+09        0.009134
2   metric  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  4.483383e+09                      30              2                                2      2022-03-20 22:28:00 CST   4.506072e+09        0.005061
3   metric  adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  9.564433e+10                      30              2                                2      2022-03-20 22:28:00 CST   9.589333e+10        0.002603
4   metric   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  6.422520e+10                      30              2                                2      2022-03-20 22:28:00 CST   6.424679e+10        0.000336
5   metric   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  8.590390e+11                      30              2                                2      2022-03-20 22:28:00 CST   8.592683e+11        0.000267
6   metric      node-6                                   system.disk.used  4.388252e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.389392e+09        0.000260
7   metric      node-6                              system.fs.inodes.free  4.966671e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.967720e+09        0.000211
8   metric      node-6                             system.fs.inodes.total  4.968760e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.969808e+09        0.000211
9   metric      node-3                              system.fs.inodes.free  4.966805e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.967566e+09        0.000153
10  metric      node-3                             system.fs.inodes.total  4.968752e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.969514e+09        0.000153
11  metric      node-4                              system.fs.inodes.free  6.609691e+09                      30              2                                2      2022-03-20 22:00:00 CST   6.610534e+09        0.000128
12  metric      node-4                             system.fs.inodes.total  6.612278e+09                      30              2                                2      2022-03-20 22:00:00 CST   6.613120e+09        0.000127
13  metric      node-1                              system.fs.inodes.free  4.967091e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.967635e+09        0.000110
14  metric      node-1                             system.fs.inodes.total  4.969030e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.969574e+09        0.000109
15  metric      node-2                             system.fs.inodes.total  4.969030e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.969565e+09        0.000108
16  metric      node-2                              system.fs.inodes.free  4.967061e+09                      30              2                                2      2022-03-20 22:00:00 CST   4.967596e+09        0.000108
17  metric   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  7.886088e+10                      30              2                                1      2022-03-20 22:27:00 CST   7.886500e+10        0.000052
18  metric   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  8.107171e+10                      30              2                                1      2022-03-20 22:27:00 CST   8.107497e+10        0.000040
19  metric      node-4                                   system.disk.used  5.768349e+09                      30              2                                1      2022-03-20 22:25:00 CST   5.768359e+09        0.000002, 'No multi-minute consecutive faults (>=3 minutes) found among top 20 candidates.')```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse timezone and (if present) previously loaded err_df, start_ts, end_ts; otherwise load/compute.
tz = pytz.timezone('Asia/Shanghai')

try:
    err_df  # use existing if available
except NameError:
    err_df = pd.read_csv('error_logs.csv')

# Ensure timestamp int
err_df['timestamp'] = err_df['timestamp'].astype(int)

# Compute incident window bounds (Asia/Shanghai) — recalculated for safety
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Filter to incident window and selected cmdb_ids
selected_cmdbs = ["frontend-2","frontend-0","frontend-1"]
err_win_sel = err_df[
    (err_df['timestamp'] >= start_ts) & 
    (err_df['timestamp'] <= end_ts) & 
    (err_df['cmdb_id'].isin(selected_cmdbs))
].copy()

# A) Counts summary: cmdb_id, count, min_timestamp, max_timestamp sorted by count desc
counts_summary = (
    err_win_sel.groupby('cmdb_id', dropna=False)
    .agg(count=('timestamp','count'),
         min_timestamp=('timestamp','min'),
         max_timestamp=('timestamp','max'))
    .reset_index()
)
# Format timestamps to Asia/Shanghai strings
counts_summary['min_timestamp'] = pd.to_datetime(counts_summary['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
counts_summary['max_timestamp'] = pd.to_datetime(counts_summary['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
counts_summary = counts_summary.sort_values('count', ascending=False).reset_index(drop=True)

# B) Sample messages table: up to 50 rows sorted by cmdb_id desc then timestamp asc
sample_messages = (
    err_win_sel[['timestamp','cmdb_id','message']]
    .sort_values(['cmdb_id','timestamp'], ascending=[False, True])
    .copy()
)
# Format timestamp
sample_messages['timestamp'] = pd.to_datetime(sample_messages['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
sample_messages = sample_messages.head(50).reset_index(drop=True)

# Display compact outputs (counts_summary and sample_messages)
counts_summary, sample_messages

```
Out[4]:
```
```python
# Summarize the filtered error_logs results in plain English using existing session variables.
# Display a single compact summary string.

# Reuse counts_summary and sample_messages from the kernel
top_counts = counts_summary  # already sorted by count desc
samples = sample_messages    # up to 50 rows

lines = []
lines.append("Error logs (incident window 2022-03-20 22:00:00 CST to 2022-03-20 22:29:00 CST) — selected frontends:")
for row in top_counts.itertuples(index=False):
    lines.append(f"- {row.cmdb_id}: count={int(row.count)}, earliest={row.min_timestamp}, latest={row.max_timestamp}")

# Inspect sample message patterns (keep short)
unique_msgs = samples['message'].dropna().unique()[:3]
msg_sample_text = "; ".join([m if len(m)<120 else m[:117]+"..." for m in unique_msgs])

lines.append("")
lines.append(f"Sample messages (up to 50 rows shown, sorted by cmdb_id desc then timestamp asc).")
lines.append(f"Typical message patterns include: {msg_sample_text}")
lines.append("")
lines.append("Summary:")
lines.append("- frontend-2 has the highest error volume (443 messages) followed by frontend-0 (144) and frontend-1 (73).")
lines.append("- Messages are concentrated from ~22:00 to ~22:29 with many repeated warning entries on frontend-2.")
lines.append("- Use the sample messages table for concrete message content and timestamps (up to 50 rows).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count            min_timestamp            max_timestamp
0  frontend-2    443  2022-03-20 22:00:01 CST  2022-03-20 22:28:56 CST
1  frontend-0    144  2022-03-20 22:00:07 CST  2022-03-20 22:28:46 CST
2  frontend-1     73  2022-03-20 22:00:16 CST  2022-03-20 22:28:38 CST,                   timestamp     cmdb_id                                            message
0   2022-03-20 22:00:01 CST  frontend-2  severity: warning, message: failed to retrieve...
1   2022-03-20 22:00:02 CST  frontend-2  severity: warning, message: failed to retrieve...
2   2022-03-20 22:00:07 CST  frontend-2  severity: warning, message: failed to retrieve...
3   2022-03-20 22:00:12 CST  frontend-2  severity: warning, message: failed to retrieve...
4   2022-03-20 22:00:17 CST  frontend-2  severity: warning, message: failed to retrieve...
..                      ...         ...                                                ...
45  2022-03-20 22:03:06 CST  frontend-2  severity: warning, message: failed to retrieve...
46  2022-03-20 22:03:08 CST  frontend-2  severity: warning, message: failed to retrieve...
47  2022-03-20 22:03:11 CST  frontend-2  severity: warning, message: failed to retrieve...
48  2022-03-20 22:03:17 CST  frontend-2  severity: warning, message: failed to retrieve...
49  2022-03-20 22:03:19 CST  frontend-2  severity: warning, message: failed to retrieve...

[50 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# --- Load trace.csv fresh to ensure correct column names ---
trace_full = pd.read_csv('trace.csv')
# Ensure expected columns exist
expected = {'timestamp','cmdb_id','trace_name','value'}
if not expected.issubset(set(trace_full.columns)):
    raise ValueError(f"trace.csv missing expected columns: {expected - set(trace_full.columns)}")

# Normalize types
trace_full['timestamp'] = trace_full['timestamp'].astype(int)
trace_full['value'] = pd.to_numeric(trace_full['value'], errors='coerce')

# --- 1) Compute global P95 per (cmdb_id, trace_name) using full series ---
trace_thresholds = (
    trace_full
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'p95'})
)
# keep compact numeric formatting later

# --- 2) Filter to incident window inclusive ---
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

trace_win = trace_full[(trace_full['timestamp'] >= start_ts) & (trace_full['timestamp'] <= end_ts)].copy()

# --- 3) Join filtered rows with thresholds and aggregate ---
merged = trace_win.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
# Drop rows without threshold (if any) because we can't compute anomalies without p95
merged = merged.dropna(subset=['p95'])

# Anomaly flag
merged['is_anomaly'] = merged['value'] > merged['p95']

# Aggregations
agg = merged.groupby(['cmdb_id','trace_name'], dropna=False).agg(
    p95 = ('p95', 'first'),
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    mean_value_in_window = ('value','mean')
).reset_index()

# earliest_anomaly_timestamp: compute separately and merge
ea = merged[merged['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
if not ea.empty:
    # format timestamps to Asia/Shanghai
    ea['earliest_anomaly_timestamp'] = pd.to_datetime(ea['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg = agg.merge(ea, on=['cmdb_id','trace_name'], how='left')

# Format numeric columns compactly
agg['p95'] = agg['p95'].astype(float).round(6)
agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
agg['mean_value_in_window'] = agg['mean_value_in_window'].astype(float).round(6)
agg['anomaly_count'] = agg['anomaly_count'].astype(int)
agg['total_points_in_window'] = agg['total_points_in_window'].astype(int)

# Ensure earliest_anomaly_timestamp NA shown as <NA>
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].fillna(pd.NA)

# --- 4) Top 20 sorted by anomaly_count desc then max_value_in_window desc ---
trace_anomalies_top20 = agg.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# Select and order columns as requested
trace_anomalies_top20 = trace_anomalies_top20[[
    'cmdb_id','trace_name','p95','total_points_in_window','anomaly_count',
    'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
]]

# --- 5) Summary: top 5 cmdb_id by total trace rows in the window ---
top_cmdb_counts = trace_win.groupby('cmdb_id', dropna=False).size().rename('count').reset_index()
top5 = top_cmdb_counts.sort_values('count', ascending=False).head(5)
top5_summary = ", ".join([f"{r.cmdb_id}:{int(r.count)}" for r in top5.itertuples(index=False)])

# Final compact outputs
trace_anomalies_top20, top5_summary

```
Out[5]:
```
```python
# Summarize trace-level anomaly detection results in plain English using existing kernel variables.
top_rows = trace_anomalies_top20.head(5)

lines = []
lines.append("Incident window: 2022-03-20 22:00:00 CST to 2022-03-20 22:29:00 CST.")
lines.append("")
lines.append("Top trace anomaly entries (compact):")
for r in top_rows.itertuples(index=False):
    lines.append(f"- {r.cmdb_id} | {r.trace_name} | p95={r.p95} | points={int(r.total_points_in_window)} | anomalies={int(r.anomaly_count)} | earliest={r.earliest_anomaly_timestamp} | max={r.max_value_in_window}")

lines.append("")
lines.append("Top 5 components by trace volume in window:")
lines.append(top5_summary)
lines.append("")
lines.append("Notes:")
lines.append("- Most top trace entries have 2 anomaly points in the window and short-lived peaks (examples earliest anomalies around 22:03–22:10 CST).")
lines.append("- Frontend pods (frontend-2, frontend-0, frontend-1) dominate both anomaly counts and trace volume.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                   trace_name     p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0                 frontend-2                         trace.self.row_count  852.55                      30              2    2022-03-20 22:03:00 CST                865.0            766.433333
1                 frontend-0                         trace.self.row_count  282.65                      30              2    2022-03-20 22:06:00 CST                300.0            248.300000
2                frontend2-0                         trace.self.row_count  153.40                      30              2    2022-03-20 22:05:00 CST                273.0            127.633333
3                 frontend-1                         trace.self.row_count  161.55                      30              2    2022-03-20 22:10:00 CST                196.0            128.166667
4                 frontend-2   trace.to_productcatalogservice-1.row_count  132.65                      30              2    2022-03-20 22:03:00 CST                137.0            118.233333
5    productcatalogservice-1              trace.from_frontend-2.row_count  132.65                      30              2    2022-03-20 22:03:00 CST                137.0            118.233333
6                 frontend-2   trace.to_productcatalogservice-0.row_count  132.65                      30              2    2022-03-20 22:03:00 CST                136.0            118.233333
7                 frontend-2   trace.to_productcatalogservice-2.row_count  132.20                      30              2    2022-03-20 22:03:00 CST                136.0            118.233333
8    productcatalogservice-0              trace.from_frontend-2.row_count  132.65                      30              2    2022-03-20 22:03:00 CST                136.0            118.233333
9    productcatalogservice-2              trace.from_frontend-2.row_count  132.20                      30              2    2022-03-20 22:03:00 CST                136.0            118.233333
10               frontend2-0  trace.to_productcatalogservice2-0.row_count   78.95                      30              2    2022-03-20 22:05:00 CST                128.0             58.600000
11  productcatalogservice2-0             trace.from_frontend2-0.row_count   78.95                      30              2    2022-03-20 22:05:00 CST                128.0             58.600000
12         currencyservice-1              trace.from_frontend-2.row_count   85.10                      30              2    2022-03-20 22:10:00 CST                 91.0             74.966667
13                frontend-2         trace.to_currencyservice-1.row_count   85.10                      30              2    2022-03-20 22:10:00 CST                 91.0             74.966667
14                frontend-2                    trace.from_root.row_count   85.20                      30              2    2022-03-20 22:03:00 CST                 87.0             76.900000
15        currencyservice2-0             trace.from_frontend2-0.row_count   52.75                      30              2    2022-03-20 22:05:00 CST                 78.0             37.966667
16               frontend2-0        trace.to_currencyservice2-0.row_count   52.75                      30              2    2022-03-20 22:05:00 CST                 78.0             37.966667
17                frontend-0   trace.to_productcatalogservice-1.row_count   46.10                      30              2    2022-03-20 22:26:00 CST                 50.0             38.200000
18                frontend-0   trace.to_productcatalogservice-2.row_count   46.10                      30              2    2022-03-20 22:26:00 CST                 50.0             38.200000
19   productcatalogservice-1              trace.from_frontend-0.row_count   46.10                      30              2    2022-03-20 22:26:00 CST                 50.0             38.200000, 'frontend-2:2720, frontend-0:2508, frontend-1:2344, checkoutservice-1:1940, checkoutservice-0:1900')```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing variables if present, otherwise load
tz = pytz.timezone('Asia/Shanghai')

try:
    trace_full
except NameError:
    trace_full = pd.read_csv('trace.csv')

# Ensure types
trace_full['timestamp'] = trace_full['timestamp'].astype(int)
trace_full['value'] = pd.to_numeric(trace_full['value'], errors='coerce')
trace_full['trace_name'] = trace_full['trace_name'].astype(str)
trace_full['cmdb_id'] = trace_full['cmdb_id'].astype(str)

# Ensure incident window bounds exist (reuse if available)
try:
    start_ts, end_ts
except NameError:
    start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
    end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
    start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
    end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# 1) Identify duration-related trace rows and compute global P95 per (cmdb_id, trace_name)
duration_mask = trace_full['trace_name'].str.contains('duration', case=False, na=False)
trace_duration_full = trace_full[duration_mask].copy()

# Compute global P95 thresholds BEFORE filtering
trace_duration_thresholds = (
    trace_duration_full
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'p95'})
)

# 2) Filter duration-related rows to incident window (inclusive)
trace_duration_win = trace_duration_full[
    (trace_duration_full['timestamp'] >= start_ts) & 
    (trace_duration_full['timestamp'] <= end_ts)
].copy()

# 3) Join filtered rows with thresholds and aggregate required metrics
merged = trace_duration_win.merge(trace_duration_thresholds, on=['cmdb_id','trace_name'], how='left')
# Drop rows without p95 (can't assess anomaly)
merged = merged.dropna(subset=['p95'])

merged['is_anomaly'] = merged['value'] > merged['p95']

# Aggregations
agg = merged.groupby(['cmdb_id','trace_name'], dropna=False).agg(
    p95 = ('p95','first'),
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    mean_value_in_window = ('value','mean')
).reset_index()

# earliest_anomaly_timestamp
ea = merged[merged['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
if not ea.empty:
    ea['earliest_anomaly_timestamp'] = pd.to_datetime(ea['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg = agg.merge(ea, on=['cmdb_id','trace_name'], how='left')

# Format numeric columns compactly and fill NAs
agg['p95'] = agg['p95'].astype(float).round(6)
agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
agg['mean_value_in_window'] = agg['mean_value_in_window'].astype(float).round(6)
agg['anomaly_count'] = agg['anomaly_count'].astype(int)
agg['total_points_in_window'] = agg['total_points_in_window'].astype(int)
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].fillna(pd.NA)

# 4) Top 20 sorted by anomaly_count desc then max_value_in_window desc
duration_anomalies_top20 = agg.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)
duration_anomalies_top20 = duration_anomalies_top20[[
    'cmdb_id','trace_name','p95','total_points_in_window','anomaly_count',
    'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
]]

# 5) Top 5 cmdb_id by number of duration-trace rows in the window
top_cmdb_counts = trace_duration_win.groupby('cmdb_id', dropna=False).size().rename('count').reset_index()
top5 = top_cmdb_counts.sort_values('count', ascending=False).head(5)
duration_top5_summary = ", ".join([f"{r.cmdb_id}:{int(r.count)}" for r in top5.itertuples(index=False)])

# Display compact outputs
duration_anomalies_top20, duration_top5_summary

```
Out[6]:
```
```python
# Summarize duration-related trace anomaly results from the session in plain English.
# Reuse existing variables: duration_anomalies_top20, duration_top5_summary

try:
    sample_rows = duration_anomalies_top20.head(5)
    top5_comp = duration_top5_summary
except NameError:
    # Fallback message if variables are not present
    summary = "Duration-trace analysis results not available in the current session."
else:
    lines = []
    lines.append("Duration-trace anomaly summary for incident window 2022-03-20 22:00–22:29 CST:")
    lines.append("")
    lines.append("Key findings (top duration-related trace entries):")
    for r in sample_rows.itertuples(index=False):
        lines.append(
            f"- {r.cmdb_id} | {r.trace_name} | p95={r.p95} | points={int(r.total_points_in_window)} | "
            f"anomalies={int(r.anomaly_count)} | earliest={r.earliest_anomaly_timestamp} | max={r.max_value_in_window}"
        )
    lines.append("")
    lines.append("Summary observations:")
    lines.append("- Most flagged duration traces have 2 anomaly points in the 30-minute window (short-lived spikes).")
    lines.append("- Total points in-window are typically 30 (some services slightly less, e.g., 25–29).")
    lines.append("- Earliest anomaly times vary across services (examples range ~22:00 to ~22:21 CST).")
    lines.append("- The p95 vs max differences are small in absolute terms (examples show marginal exceedances).")
    lines.append("")
    lines.append("Top components by number of duration trace rows in the window (cmdb_id:count):")
    lines.append(top5_comp)
    lines.append("")
    lines.append("Conclusion: duration-related traces show brief, low-volume spikes at multiple frontend and checkout-related components; no large sustained duration outages are evident among the top duration-trace candidates.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                      trace_name       p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0                frontend2-0                    trace.from_root.duration_p95  0.327055                      30              2    2022-03-20 22:02:00 CST             0.342486              0.166566
1                frontend2-0                   trace.from_root.duration_mean  0.147239                      30              2    2022-03-20 22:00:00 CST             0.160214              0.098884
2          checkoutservice-1             trace.from_frontend-2.duration_mean  0.055855                      29              2    2022-03-20 22:14:00 CST             0.108685              0.041155
3          checkoutservice-1              trace.from_frontend-2.duration_p95  0.055855                      29              2    2022-03-20 22:14:00 CST             0.108685              0.041313
4                 frontend-2        trace.to_checkoutservice-1.duration_mean  0.055855                      29              2    2022-03-20 22:14:00 CST             0.108685              0.041155
5                 frontend-2         trace.to_checkoutservice-1.duration_p95  0.055855                      29              2    2022-03-20 22:14:00 CST             0.108685              0.041313
6                 frontend-1                    trace.from_root.duration_p95  0.073285                      30              2    2022-03-20 22:04:00 CST             0.080663              0.059710
7                 frontend-0                    trace.from_root.duration_p95  0.076927                      30              2    2022-03-20 22:25:00 CST             0.077234              0.059117
8                frontend2-0  trace.to_recommendationservice2-0.duration_p95  0.064035                      30              2    2022-03-20 22:20:00 CST             0.077229              0.013400
9   recommendationservice2-0             trace.from_frontend2-0.duration_p95  0.064035                      30              2    2022-03-20 22:20:00 CST             0.077229              0.013400
10  recommendationservice2-0                         trace.self.duration_p95  0.063664                      30              2    2022-03-20 22:20:00 CST             0.076871              0.011965
11                frontend-2                    trace.from_root.duration_p95  0.076115                      30              2    2022-03-20 22:21:00 CST             0.076721              0.062846
12         checkoutservice-2              trace.from_frontend-2.duration_p95  0.064989                      25              2    2022-03-20 22:12:00 CST             0.075145              0.041897
13                frontend-2         trace.to_checkoutservice-2.duration_p95  0.064989                      25              2    2022-03-20 22:12:00 CST             0.075145              0.041897
14         checkoutservice-0             trace.from_frontend-2.duration_mean  0.067535                      26              2    2022-03-20 22:15:00 CST             0.073120              0.039405
15         checkoutservice-0              trace.from_frontend-2.duration_p95  0.071160                      26              2    2022-03-20 22:15:00 CST             0.073120              0.040260
16                frontend-2        trace.to_checkoutservice-0.duration_mean  0.067535                      26              2    2022-03-20 22:15:00 CST             0.073120              0.039405
17                frontend-2         trace.to_checkoutservice-0.duration_p95  0.071160                      26              2    2022-03-20 22:15:00 CST             0.073120              0.040260
18         checkoutservice-1                         trace.self.duration_p95  0.015325                      29              2    2022-03-20 22:11:00 CST             0.071579              0.010091
19               frontend2-0                         trace.self.duration_p95  0.064705                      30              2    2022-03-20 22:21:00 CST             0.068976              0.051037, 'frontend-2:1360, frontend-0:1254, frontend-1:1172, checkoutservice-1:970, checkoutservice-0:950')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Load err_df if not already in session
try:
    err_df
except NameError:
    err_df = pd.read_csv('error_logs.csv')

# Ensure timestamp int
err_df['timestamp'] = err_df['timestamp'].astype(int)
err_df['cmdb_id'] = err_df['cmdb_id'].astype(str)
err_df['message'] = err_df['message'].astype(str)

# Incident window bounds (Asia/Shanghai -> UTC)
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Filter window and selected cmdb_ids
selected = ["frontend-2","frontend-0","frontend-1"]
err_win_sel = err_df[
    (err_df['timestamp'] >= start_ts) &
    (err_df['timestamp'] <= end_ts) &
    (err_df['cmdb_id'].isin(selected))
].copy()

# Drop rows with empty message (if any)
err_win_sel = err_win_sel[err_win_sel['message'].notna()]

# Group by full message string and compute required aggregates
grp = (
    err_win_sel
    .groupby('message', dropna=False)
    .agg(
        count = ('timestamp','count'),
        cmdb_ids = ('cmdb_id', lambda s: ",".join(sorted(s.unique()))),
        min_ts = ('timestamp','min'),
        max_ts = ('timestamp','max')
    )
    .reset_index()
)

# Format message_sample (first 200 chars) and timestamps to Asia/Shanghai strings
grp['message_sample'] = grp['message'].str.slice(0,200)
grp['earliest_timestamp'] = pd.to_datetime(grp['min_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
grp['latest_timestamp']   = pd.to_datetime(grp['max_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order requested columns, sort by count desc, limit top 10
counts_top10 = grp.sort_values('count', ascending=False)[['message_sample','cmdb_ids','count','earliest_timestamp','latest_timestamp']].head(10).rename(columns={'cmdb_ids':'cmdb_id_list'})

# For the top-ranked message pattern, get up to 10 example rows sorted by timestamp asc
if not counts_top10.empty:
    top_message_full = grp.sort_values('count', ascending=False).iloc[0]['message']
    examples = err_win_sel[err_win_sel['message'] == top_message_full].copy()
    examples = examples.sort_values('timestamp', ascending=True)[['timestamp','cmdb_id','message']].head(10)
    # Format timestamp to Asia/Shanghai strings
    examples['timestamp'] = pd.to_datetime(examples['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    top_examples = examples.rename(columns={'message':'full_message'}).reset_index(drop=True)
else:
    top_examples = pd.DataFrame(columns=['timestamp','cmdb_id','full_message'])

# Output the two compact tables
counts_top10, top_examples

```
Out[7]:
```
```python
# Summarize the grouped error-log message results in plain English using existing session variables.
try:
    top_row = counts_top10.iloc[0]
    examples = top_examples.head(5)
    msg_sample = top_row['message_sample']
    cmdb_list = top_row['cmdb_id_list']
    total_count = int(top_row['count'])
    earliest = top_row['earliest_timestamp']
    latest = top_row['latest_timestamp']

    example_lines = []
    for r in examples.itertuples(index=False):
        example_lines.append(f"{r.timestamp} {r.cmdb_id}")

    summary = (
        "Error-log message pattern summary (incident window 2022-03-20 22:00–22:29 CST):\n\n"
        f"Top message (first 200 chars): \"{msg_sample}\"\n"
        f"Emitted by: {cmdb_list}\n"
        f"Total occurrences (selected frontends): {total_count}\n"
        f"Earliest occurrence: {earliest}\n"
        f"Latest occurrence: {latest}\n\n"
        "Sample occurrences (up to 5, timestamp and cmdb_id):\n"
        + "\n".join(example_lines) + "\n\n"
        "Interpretation: The dominant message is a repeated warning ('failed to retrieve...') emitted by all three frontends during the incident window, with frontend-2 producing many of the earliest events. This looks like a widespread, repeated frontend warning occurring throughout the 30-minute window."
    )
except Exception as e:
    summary = "No error-log summary available in the current session."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                      message_sample                      cmdb_id_list  count       earliest_timestamp         latest_timestamp
0  severity: warning, message: failed to retrieve...  frontend-0,frontend-1,frontend-2    660  2022-03-20 22:00:01 CST  2022-03-20 22:28:56 CST,                  timestamp     cmdb_id                                       full_message
0  2022-03-20 22:00:01 CST  frontend-2  severity: warning, message: failed to retrieve...
1  2022-03-20 22:00:02 CST  frontend-2  severity: warning, message: failed to retrieve...
2  2022-03-20 22:00:07 CST  frontend-0  severity: warning, message: failed to retrieve...
3  2022-03-20 22:00:07 CST  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-20 22:00:12 CST  frontend-2  severity: warning, message: failed to retrieve...
5  2022-03-20 22:00:16 CST  frontend-0  severity: warning, message: failed to retrieve...
6  2022-03-20 22:00:16 CST  frontend-1  severity: warning, message: failed to retrieve...
7  2022-03-20 22:00:17 CST  frontend-2  severity: warning, message: failed to retrieve...
8  2022-03-20 22:00:22 CST  frontend-2  severity: warning, message: failed to retrieve...
9  2022-03-20 22:00:36 CST  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing data where possible; otherwise load trace.csv
try:
    trace_full
except NameError:
    trace_full = pd.read_csv('trace.csv')

# Ensure types
trace_full['timestamp'] = trace_full['timestamp'].astype(int)
trace_full['value'] = pd.to_numeric(trace_full['value'], errors='coerce')
trace_full['cmdb_id'] = trace_full['cmdb_id'].astype(str)
trace_full['trace_name'] = trace_full['trace_name'].astype(str)

# Ensure global trace thresholds exist (computed on full series BEFORE filtering)
try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        trace_full
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'p95'})
    )

# Define timezone and incident window (Asia/Shanghai UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Filter trace rows for frontend-2 within the incident window
fw = trace_full[
    (trace_full['cmdb_id'] == 'frontend-2') &
    (trace_full['timestamp'] >= start_ts) &
    (trace_full['timestamp'] <= end_ts)
].copy()

# Join with precomputed global P95 thresholds on (cmdb_id, trace_name)
fw = fw.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Drop rows without p95 (cannot evaluate anomaly)
fw = fw.dropna(subset=['p95'])

# Identify anomalies (value > p95)
fw['is_anomaly'] = fw['value'] > fw['p95']

# Aggregate per trace_name where any anomaly in window
if not fw.empty:
    ea = fw[fw['is_anomaly']].groupby('trace_name', dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_ts'})
    agg = fw.groupby('trace_name', dropna=False).agg(
        p95 = ('p95','first'),
        total_points_in_window = ('value','count'),
        anomaly_count = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        mean_value_in_window = ('value','mean')
    ).reset_index()
    agg = agg.merge(ea, on='trace_name', how='left')
    # Keep only trace_names with anomaly_count > 0
    agg = agg[agg['anomaly_count'] > 0].copy()
    if not agg.empty:
        # Format fields
        agg['p95'] = agg['p95'].astype(float).round(6)
        agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
        agg['mean_value_in_window'] = agg['mean_value_in_window'].astype(float).round(6)
        agg['total_points_in_window'] = agg['total_points_in_window'].astype(int)
        agg['anomaly_count'] = agg['anomaly_count'].astype(int)
        # Format earliest_anomaly_timestamp to Asia/Shanghai string
        agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        # Select requested columns and sort by earliest_anomaly_timestamp ascending
        anoms_frontend2 = agg[[
            'trace_name','p95','total_points_in_window','anomaly_count',
            'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
        ]].sort_values('earliest_anomaly_timestamp', ascending=True).reset_index(drop=True).head(20)
        note = pd.NA
    else:
        # No anomalies found
        anoms_frontend2 = pd.DataFrame(columns=[
            'trace_name','p95','total_points_in_window','anomaly_count',
            'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
        ])
        note = 'no trace anomalies for frontend-2 in window'
else:
    anoms_frontend2 = pd.DataFrame(columns=[
        'trace_name','p95','total_points_in_window','anomaly_count',
        'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
    ])
    note = 'no trace anomalies for frontend-2 in window'

# Display compact result
anoms_frontend2, note

```
Out[8]:
```
```python
# Compact plain-English summary of trace anomalies for frontend-2 in the incident window.
summary = (
    "Trace anomalies for frontend-2 (2022-03-20 22:00–22:29 CST):\n"
    "- Multiple trace metrics for frontend-2 exceeded their global P95 during the incident window.\n"
    "- The first anomalies appear at 22:00:00 CST (e.g., trace.to_shippingservice-2/1.row_count); many other anomalies follow by ~22:01–22:03 CST.\n"
    "- Most flagged trace_names show only short bursts: anomaly_count = 2 (minor, brief exceedances) for the majority of entries.\n"
    "- Notable spikes (examples): trace.self.row_count peaked to 865 (p95=852.55) at ~22:03; productcatalogservice-related row_count traces peaked ~136–137 around 22:03.\n"
    "- Duration-related traces also had small absolute spikes (very small latency values) but were above their p95 for brief points.\n"
    "- Overall interpretation: frontend-2 experienced several short, low-duration spikes across many trace metrics concentrated at the start of the window (22:00–22:03), not a prolonged or multi-minute sustained duration outage."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                        trace_name         p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0             trace.to_shippingservice-2.row_count    6.550000                      30              2    2022-03-20 22:00:00 CST             8.000000              5.200000
1             trace.to_shippingservice-1.row_count    6.550000                      30              2    2022-03-20 22:00:00 CST             8.000000              5.200000
2          trace.to_shippingservice-2.duration_p95    0.000238                      30              2    2022-03-20 22:00:00 CST             0.000369              0.000092
3                   trace.to_adservice-1.row_count   18.550000                      30              2    2022-03-20 22:01:00 CST            19.000000             15.233333
4                   trace.to_adservice-0.row_count   18.550000                      30              2    2022-03-20 22:01:00 CST            19.000000             15.233333
5                trace.to_adservice-0.duration_p95    0.000031                      30              2    2022-03-20 22:02:00 CST             0.000051              0.000019
6         trace.to_checkoutservice-2.duration_mean    0.054489                      25              2    2022-03-20 22:02:00 CST             0.057207              0.040441
7               trace.to_adservice-0.duration_mean    0.000017                      30              2    2022-03-20 22:02:00 CST             0.000020              0.000014
8                             trace.self.row_count  852.550000                      30              2    2022-03-20 22:03:00 CST           865.000000            766.433333
9                        trace.from_root.row_count   85.200000                      30              2    2022-03-20 22:03:00 CST            87.000000             76.900000
10  trace.to_productcatalogservice-0.duration_mean    0.004323                      30              2    2022-03-20 22:03:00 CST             0.004363              0.004143
11                trace.to_cartservice-1.row_count   25.000000                      30              1    2022-03-20 22:03:00 CST            26.000000             22.500000
12      trace.to_productcatalogservice-0.row_count  132.650000                      30              2    2022-03-20 22:03:00 CST           136.000000            118.233333
13      trace.to_productcatalogservice-1.row_count  132.650000                      30              2    2022-03-20 22:03:00 CST           137.000000            118.233333
14                trace.to_cartservice-0.row_count   25.000000                      30              1    2022-03-20 22:03:00 CST            26.000000             22.500000
15      trace.to_productcatalogservice-2.row_count  132.200000                      30              2    2022-03-20 22:03:00 CST           136.000000            118.233333
16      trace.to_recommendationservice-0.row_count   21.000000                      30              1    2022-03-20 22:03:00 CST            22.000000             18.400000
17                trace.to_cartservice-2.row_count   25.000000                      30              1    2022-03-20 22:03:00 CST            26.000000             22.500000
18         trace.to_shippingservice-1.duration_p95    0.000254                      30              2    2022-03-20 22:03:00 CST             0.000305              0.000111
19        trace.to_shippingservice-1.duration_mean    0.000105                      30              2    2022-03-20 22:03:00 CST             0.000134              0.000072, <NA>)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing variables where possible
tz = pytz.timezone('Asia/Shanghai')

# Load trace_full if not present
try:
    trace_full
except NameError:
    trace_full = pd.read_csv('trace.csv')

# Ensure correct dtypes
trace_full['timestamp'] = trace_full['timestamp'].astype(int)
trace_full['value'] = pd.to_numeric(trace_full['value'], errors='coerce')
trace_full['cmdb_id'] = trace_full['cmdb_id'].astype(str)
trace_full['trace_name'] = trace_full['trace_name'].astype(str)

# Compute global P95 thresholds from full series if not already computed (rule: thresholds BEFORE filtering)
if 'trace_thresholds' not in globals():
    trace_thresholds = (
        trace_full
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'p95'})
    )

# Incident window bounds (Asia/Shanghai -> convert to UTC epoch)
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Shippingservice pods of interest
shipping_pods = ["shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0"]

# Filter trace rows to shipping pods within the incident window
trace_win_shipping = trace_full[
    (trace_full['cmdb_id'].isin(shipping_pods)) &
    (trace_full['timestamp'] >= start_ts) &
    (trace_full['timestamp'] <= end_ts)
].copy()

# Join with global P95 thresholds on (cmdb_id, trace_name)
merged = trace_win_shipping.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Drop rows missing p95 (cannot evaluate anomalies without threshold)
merged = merged.dropna(subset=['p95'])

# Flag anomalies where value > p95
merged['is_anomaly'] = merged['value'] > merged['p95']

# Aggregate required metrics per (cmdb_id, trace_name)
agg = merged.groupby(['cmdb_id','trace_name'], dropna=False).agg(
    p95 = ('p95','first'),
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    mean_value_in_window = ('value','mean')
).reset_index()

# Compute earliest anomaly timestamp per group (if any)
ea = merged[merged['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anom_ts'})
if not ea.empty:
    ea['earliest_anomaly_timestamp'] = pd.to_datetime(ea['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg = agg.merge(ea[['cmdb_id','trace_name','earliest_anomaly_timestamp']], on=['cmdb_id','trace_name'], how='left')
else:
    agg['earliest_anomaly_timestamp'] = pd.NA

# Ensure numeric formatting and types, round for compactness
agg['p95'] = agg['p95'].astype(float).round(6)
agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
agg['mean_value_in_window'] = agg['mean_value_in_window'].astype(float).round(6)
agg['total_points_in_window'] = agg['total_points_in_window'].astype(int)
agg['anomaly_count'] = agg['anomaly_count'].astype(int)
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].fillna(pd.NA)

# Sort by earliest_anomaly_timestamp ascending (NA last). Use timestamp conversion for sort keys.
def sort_key(ts):
    return pd.to_datetime(ts, format='%Y-%m-%d %H:%M:%S %Z', utc=True) if pd.notna(ts) else pd.NaT

# Create helper column for sorting
agg['_sort_ts'] = pd.to_datetime(agg['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
agg = agg.sort_values(by=['_sort_ts'], ascending=True).drop(columns=['_sort_ts']).reset_index(drop=True)

# Limit to top 20 rows for compactness
shipping_anomalies_top20 = agg[[
    'cmdb_id','trace_name','p95','total_points_in_window','anomaly_count',
    'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window'
]].head(20)

# Compute earliest_shipping_anomaly scalar (minimum earliest_anomaly_timestamp across rows)
valid_earlies = shipping_anomalies_top20['earliest_anomaly_timestamp'].dropna()
if not valid_earlies.empty:
    earliest_shipping_anomaly = valid_earlies.min()
else:
    earliest_shipping_anomaly = pd.NA

# Final compact outputs
shipping_anomalies_top20, earliest_shipping_anomaly

```
Out[9]:
```
```python
# Summarize shippingservice trace anomaly results compactly using existing session variables.
try:
    df = shipping_anomalies_top20  # from previous analysis
    earliest = earliest_shipping_anomaly
except NameError:
    summary = "Shippingservice trace anomaly results not available in session."
else:
    # Extract top affected pods and a few example trace_names
    top_pods = df['cmdb_id'].value_counts().index.tolist()
    top_pods_sample = ", ".join(top_pods[:4]) if top_pods else "none"
    example_traces = ", ".join(df['trace_name'].head(4).tolist())
    # Note about anomaly counts
    typical_anom_count = int(df['anomaly_count'].mode()[0]) if not df['anomaly_count'].empty else 0

    summary = (
        f"Shippingservice trace anomalies (window 2022-03-20 22:00–22:29 CST):\n"
        f"- Earliest anomaly observed: {earliest}\n"
        f"- Affected pods (examples): {top_pods_sample}\n"
        f"- Typical behavior: most flagged (cmdb_id,trace_name) pairs have anomaly_count = {typical_anom_count} (short, brief exceedances).\n"
        f"- Representative traces: {example_traces}\n"
        f"- Interpretation: multiple short-lived spikes across shippingservice pods (not a prolonged multi-minute outage); many anomalies occur right at the start of the window (~22:00–22:03 CST)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                  trace_name       p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0    shippingservice-2             trace.from_frontend-2.row_count  6.550000                      30              2    2022-03-20 22:00:00 CST             8.000000              5.200000
1    shippingservice-2          trace.from_frontend-2.duration_p95  0.000238                      30              2    2022-03-20 22:00:00 CST             0.000369              0.000092
2    shippingservice-1             trace.from_frontend-2.row_count  6.550000                      30              2    2022-03-20 22:00:00 CST             8.000000              5.200000
3    shippingservice-1         trace.from_frontend-1.duration_mean  0.000076                      23              2    2022-03-20 22:02:00 CST             0.000132              0.000065
4    shippingservice-1          trace.from_frontend-1.duration_p95  0.000076                      23              2    2022-03-20 22:02:00 CST             0.000132              0.000065
5    shippingservice-0          trace.from_frontend-0.duration_p95  0.000074                      30              2    2022-03-20 22:03:00 CST             0.000289              0.000070
6    shippingservice-1          trace.from_frontend-2.duration_p95  0.000254                      30              2    2022-03-20 22:03:00 CST             0.000305              0.000111
7    shippingservice-1         trace.from_frontend-2.duration_mean  0.000105                      30              2    2022-03-20 22:03:00 CST             0.000134              0.000072
8    shippingservice-2          trace.from_frontend-1.duration_p95  0.000079                      20              1    2022-03-20 22:04:00 CST             0.000083              0.000064
9    shippingservice-2         trace.from_frontend-1.duration_mean  0.000070                      20              1    2022-03-20 22:04:00 CST             0.000083              0.000062
10   shippingservice-0         trace.from_frontend-1.duration_mean  0.000103                      19              1    2022-03-20 22:05:00 CST             0.000264              0.000072
11   shippingservice-0          trace.from_frontend-1.duration_p95  0.000103                      19              1    2022-03-20 22:05:00 CST             0.000264              0.000073
12   shippingservice-0  trace.from_checkoutservice-1.duration_mean  0.000064                      23              2    2022-03-20 22:06:00 CST             0.000268              0.000066
13   shippingservice-0         trace.from_frontend-0.duration_mean  0.000068                      30              2    2022-03-20 22:07:00 CST             0.000180              0.000065
14   shippingservice-1  trace.from_checkoutservice-1.duration_mean  0.000068                      23              2    2022-03-20 22:09:00 CST             0.000177              0.000063
15   shippingservice-2         trace.from_frontend-2.duration_mean  0.000097                      30              2    2022-03-20 22:10:00 CST             0.000155              0.000068
16  shippingservice2-0         trace.from_frontend2-0.duration_p95  0.000233                      19              1    2022-03-20 22:10:00 CST             0.000274              0.000100
17   shippingservice-0             trace.from_frontend-0.row_count  2.550000                      30              2    2022-03-20 22:10:00 CST             3.000000              1.666667
18   shippingservice-0          trace.from_frontend-2.duration_p95  0.000307                      30              2    2022-03-20 22:11:00 CST             0.000349              0.000111
19   shippingservice-1          trace.from_frontend-0.duration_p95  0.000084                      28              2    2022-03-20 22:11:00 CST             0.000126              0.000066, '2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)

# 1) Compute global P95 thresholds per (cmdb_id, kpi_name) from full series (before filtering)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'p95'})
)

# 2) Filter metric rows to incident window and shippingservice pods, then join thresholds
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

shipping_pods = ["shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0"]

metric_win_shipping = metric_df[
    (metric_df['cmdb_id'].isin(shipping_pods)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Join with thresholds
merged = metric_win_shipping.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Drop rows without p95 (can't evaluate)
merged = merged.dropna(subset=['p95']).copy()

# Flag anomalies
merged['is_anomaly'] = merged['value'] > merged['p95']

# Helper to compute longest consecutive run (diff == 60s)
def longest_consecutive_run_minutes(timestamps):
    if len(timestamps) == 0:
        return 0, None, None  # run length, start_ts, run_max_value
    ts = np.sort(np.asarray(timestamps, dtype=np.int64))
    # If only one timestamp -> run length 1
    max_run = 1
    max_start_idx = 0
    cur_run = 1
    cur_start = 0
    for i in range(1, len(ts)):
        if ts[i] - ts[i-1] == 60:
            cur_run += 1
        else:
            if cur_run > max_run:
                max_run = cur_run
                max_start_idx = cur_start
            cur_run = 1
            cur_start = i
    if cur_run > max_run:
        max_run = cur_run
        max_start_idx = cur_start
    start_ts = int(ts[max_start_idx])
    return int(max_run), start_ts, ts[max_start_idx:max_start_idx+max_run]

# 3) Aggregate per (cmdb_id, kpi_name)
groups = []
grpby = merged.groupby(['cmdb_id','kpi_name'], dropna=False)
for (cmdb, kpi), g in grpby:
    total_points = int(g['value'].count())
    anomaly_count = int(g['is_anomaly'].sum())
    # earliest anomaly timestamp
    if anomaly_count > 0:
        earliest_anom_ts_val = int(g.loc[g['is_anomaly'],'timestamp'].min())
        earliest_anom_ts = pd.to_datetime(earliest_anom_ts_val, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        # longest consecutive run computation using anomaly timestamps and run max value inside that run
        anom_ts_arr = g.loc[g['is_anomaly'],'timestamp'].astype(int).to_numpy()
        run_len, run_start_ts, run_window_ts = longest_consecutive_run_minutes(anom_ts_arr)
        if run_len > 0:
            # compute max value within that run window
            run_mask = g['timestamp'].isin(run_window_ts)
            run_max_value = float(g.loc[run_mask, 'value'].max())
            earliest_run_start_ts_str = pd.to_datetime(int(run_start_ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            run_len = 0
            earliest_run_start_ts_str = pd.NA
            run_max_value = pd.NA
    else:
        earliest_anom_ts = pd.NA
        run_len = 0
        earliest_run_start_ts_str = pd.NA
        run_max_value = pd.NA
    max_value = float(g['value'].max())
    mean_value = float(g['value'].mean())
    p95_val = float(g['p95'].iloc[0])
    groups.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95': round(p95_val, 6),
        'total_points_in_window': total_points,
        'anomaly_count': anomaly_count,
        'longest_consecutive_anomaly_run_minutes': run_len,
        'earliest_anomaly_timestamp': earliest_anom_ts if anomaly_count>0 else pd.NA,
        'earliest_run_start_timestamp': earliest_run_start_ts_str,
        'max_value_in_window': round(max_value, 6),
        'mean_value_in_window': round(mean_value, 6)
    })

result_df = pd.DataFrame(groups)

# 4) Sort by earliest_anomaly_timestamp ascending (NA last), then anomaly_count desc
# Convert earliest_anomaly_timestamp to datetime for sorting; NA -> large date
def to_sort_dt(x):
    if pd.isna(x):
        return pd.Timestamp.max
    return pd.to_datetime(x)
result_df['_sort_ts'] = result_df['earliest_anomaly_timestamp'].apply(lambda x: to_sort_dt(x))
result_df = result_df.sort_values(by=['_sort_ts','anomaly_count'], ascending=[True, False]).drop(columns=['_sort_ts']).reset_index(drop=True)

# Limit to top 20 rows
shipping_metric_anomalies_top20 = result_df[[
    'cmdb_id','kpi_name','p95','total_points_in_window','anomaly_count',
    'longest_consecutive_anomaly_run_minutes','earliest_anomaly_timestamp',
    'max_value_in_window','mean_value_in_window'
]].head(20)

# 5) earliest_shipping_metric_anomaly scalar (minimum earliest_anomaly_timestamp across returned rows)
valid_earlies = shipping_metric_anomalies_top20['earliest_anomaly_timestamp'].dropna()
if not valid_earlies.empty:
    earliest_shipping_metric_anomaly = valid_earlies.min()
else:
    earliest_shipping_metric_anomaly = pd.NA

# Display compact outputs
shipping_metric_anomalies_top20, earliest_shipping_metric_anomaly

```
Out[10]:
```
```python
# Summarize shippingservice metric-level anomaly scan results in plain English.
# Reuse existing variables from the session: shipping_metric_anomalies_top20, earliest_shipping_metric_anomaly

try:
    df = shipping_metric_anomalies_top20
    earliest = earliest_shipping_metric_anomaly
except NameError:
    summary = "Shippingservice metric anomaly results are not available in the current session."
else:
    n_rows = len(df)
    # Typical anomaly_count distribution
    if n_rows > 0:
        typical_anom = int(df['anomaly_count'].mode()[0]) if not df['anomaly_count'].empty else 0
    else:
        typical_anom = 0

    # Prepare 3 representative examples (compact)
    examples = []
    for r in df.head(3).itertuples(index=False):
        examples.append(
            f"{r.cmdb_id} | {r.kpi_name} | p95={r.p95:.6g} | anomalies={int(r.anomaly_count)} | "
            f"longest_run_min={int(r.longest_consecutive_anomaly_run_minutes)} | earliest={r.earliest_anomaly_timestamp} | max={r.max_value_in_window:.6g}"
        )
    examples_text = "\n  ".join(examples) if examples else "none"

    summary = (
        "Shippingservice metric anomaly summary (window 2022-03-20 22:00–22:29 CST):\n\n"
        f"- Earliest shipping metric anomaly observed: {earliest}\n"
        f"- Number of (cmdb_id, kpi_name) pairs flagged (returned): {n_rows}\n"
        f"- Typical anomaly_count per pair: {typical_anom} (most pairs show short, brief exceedances)\n"
        f"- Longest consecutive anomaly runs observed are short (mostly 1–2 minutes).\n\n"
        "Representative examples:\n  " + examples_text + "\n\n"
        "Interpretation: shippingservice pods experienced multiple short, low-duration metric exceedances across CPU/memory/mesh metrics concentrated near the start of the window (many entries around 22:00–22:05). These look like brief spikes rather than sustained metric failures."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name           p95  total_points_in_window  anomaly_count  longest_consecutive_anomaly_run_minutes earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0    shippingservice-2  mesh.source.shippingservice.jaeger-collector.i...  1.600000e-03                      30              2                                        2    2022-03-20 22:00:00 CST         2.050000e-03          1.720000e-04
1    shippingservice-0      container.node-5.container_cpu_system_seconds  1.000000e-02                      30              2                                        1    2022-03-20 22:03:00 CST         1.000000e-02          3.500000e-03
2    shippingservice-0       container.node-5.container_cpu_usage_seconds  1.511300e-02                      30              2                                        1    2022-03-20 22:03:00 CST         1.647500e-02          1.200600e-02
3    shippingservice-0  container.node-5.container_memory_failures.con...  4.587500e+01                      30              2                                        1    2022-03-20 22:03:00 CST         7.700000e+01          2.086667e+01
4    shippingservice-0  container.node-5.container_memory_failures.hie...  4.587500e+01                      30              2                                        1    2022-03-20 22:03:00 CST         7.700000e+01          2.086667e+01
5    shippingservice-1  mesh.destination.checkoutservice.shippingservi...  3.088125e+03                      30              2                                        1    2022-03-20 22:03:00 CST         3.262500e+03          2.204583e+03
6    shippingservice-1  mesh.destination.checkoutservice.shippingservi...  5.343750e+03                      30              2                                        1    2022-03-20 22:03:00 CST         5.625000e+03          3.812500e+03
7    shippingservice-2              container.node-5.container_memory_rss  1.414144e+07                      30              2                                        2    2022-03-20 22:03:00 CST         1.426637e+07          1.377512e+07
8    shippingservice-2         container.node-5.container_memory_usage_MB  4.620117e+01                      30              2                                        2    2022-03-20 22:03:00 CST         4.632031e+01          4.585182e+01
9    shippingservice-2   container.node-5.container_memory_working_set_MB  4.059961e+01                      30              2                                        2    2022-03-20 22:03:00 CST         4.071875e+01          4.025026e+01
10   shippingservice-2  container.node-5.container_network_transmit_MB...  5.546790e-01                      30              2                                        1    2022-03-20 22:03:00 CST         5.574870e-01          4.554740e-01
11   shippingservice-1  container.node-5.container_cpu_cfs_throttled_s...  0.000000e+00                      30              2                                        2    2022-03-20 22:04:00 CST         0.000000e+00          0.000000e+00
12  shippingservice2-0      container.node-5.container_cpu_system_seconds  5.000000e-03                      30              1                                        1    2022-03-20 22:04:00 CST         1.000000e-02          1.333000e-03
13  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  2.615000e+03                      30              1                                        1    2022-03-20 22:04:00 CST         2.942500e+03          1.651500e+03
14  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  4.000000e+00                      30              1                                        1    2022-03-20 22:04:00 CST         4.500000e+00          2.533333e+00
15  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  4.000000e+00                      30              1                                        1    2022-03-20 22:04:00 CST         4.500000e+00          2.533333e+00
16  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  5.000000e+03                      30              1                                        1    2022-03-20 22:04:00 CST         5.625000e+03          3.166667e+03
17  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  4.000000e+00                      30              1                                        1    2022-03-20 22:04:00 CST         4.500000e+00          2.533333e+00
18   shippingservice-1       container.node-5.container_cpu_usage_seconds  1.686800e-02                      30              2                                        1    2022-03-20 22:05:00 CST         1.732100e-02          1.197200e-02
19   shippingservice-2       container.node-5.container_cpu_usage_seconds  1.743700e-02                      30              2                                        1    2022-03-20 22:05:00 CST         1.989600e-02          1.246000e-02, '2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[11]:

# Revised aggregation with robust sorting (avoid tz-aware vs tz-naive comparison issues)

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df and metric_thresholds from session (they should exist)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

if 'metric_thresholds' not in globals():
    metric_thresholds = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'p95'})
    )

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)

# Incident window bounds
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Filter to node-5 within window
node5_win = metric_df[
    (metric_df['cmdb_id'] == 'node-5') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Join with global thresholds
node5_merged = node5_win.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
node5_merged = node5_merged.dropna(subset=['p95']).copy()
node5_merged['is_anomaly'] = node5_merged['value'] > node5_merged['p95']

# Helper for longest consecutive run where diffs == 60s
def longest_run_info(timestamps):
    if len(timestamps) == 0:
        return 0, None
    ts = np.sort(np.asarray(timestamps, dtype=np.int64))
    max_run = 1
    max_start_idx = 0
    cur_run = 1
    cur_start = 0
    for i in range(1, len(ts)):
        if ts[i] - ts[i-1] == 60:
            cur_run += 1
        else:
            if cur_run > max_run:
                max_run = cur_run
                max_start_idx = cur_start
            cur_run = 1
            cur_start = i
    if cur_run > max_run:
        max_run = cur_run
        max_start_idx = cur_start
    start_ts = int(ts[max_start_idx])
    return int(max_run), start_ts

rows = []
grp = node5_merged.groupby('kpi_name', dropna=False)
for kpi, g in grp:
    total_points = int(g['value'].count())
    anomaly_count = int(g['is_anomaly'].sum())
    p95_val = float(g['p95'].iloc[0])
    max_value = float(g['value'].max())
    mean_value = float(g['value'].mean())
    if anomaly_count > 0:
        earliest_anom_ts_val = int(g.loc[g['is_anomaly'],'timestamp'].min())
        earliest_anom_str = pd.to_datetime(earliest_anom_ts_val, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        anom_ts = g.loc[g['is_anomaly'],'timestamp'].astype(int).to_numpy()
        run_len, run_start_ts = longest_run_info(anom_ts)
    else:
        earliest_anom_str = pd.NA
        run_len = 0
        run_start_ts = None
    # severity ratio
    severity_ratio = float((max_value - p95_val) / p95_val) if p95_val != 0 else float('nan')
    rows.append({
        'cmdb_id': 'node-5',
        'kpi_name': kpi,
        'p95': round(p95_val, 6),
        'total_points_in_window': total_points,
        'anomaly_count': anomaly_count,
        'longest_consecutive_anomaly_run_minutes': run_len,
        'earliest_anomaly_timestamp': earliest_anom_str,
        'max_value_in_window': round(max_value, 6),
        'severity_ratio': severity_ratio
    })

result_df = pd.DataFrame(rows)

# Robust sorting: convert earliest_anomaly_timestamp to epoch seconds (NaN -> large number)
def to_epoch_or_large(ts_str):
    if pd.isna(ts_str):
        return 9999999999
    return int(pd.to_datetime(ts_str, utc=True).timestamp())

result_df['_epoch_sort'] = result_df['earliest_anomaly_timestamp'].apply(lambda x: to_epoch_or_large(x))
result_df = result_df.sort_values(by=['_epoch_sort','severity_ratio'], ascending=[True, False]).drop(columns=['_epoch_sort']).reset_index(drop=True)

# Limit to top 20 rows
node5_anomalies_top20 = result_df[[
    'cmdb_id','kpi_name','p95','total_points_in_window','anomaly_count',
    'longest_consecutive_anomaly_run_minutes','earliest_anomaly_timestamp','max_value_in_window','severity_ratio'
]].head(20)

# earliest_node5_anomaly scalar
valid_earlies = node5_anomalies_top20['earliest_anomaly_timestamp'].dropna()
earliest_node5_anomaly = valid_earlies.min() if not valid_earlies.empty else pd.NA

# Outputs (compact)
node5_anomalies_top20, earliest_node5_anomaly

```
Out[11]:
```
```python
# Summarize node-5 metric anomaly findings in plain English using existing session variables.
try:
    df = node5_anomalies_top20
    earliest = earliest_node5_anomaly
except NameError:
    summary = "Node-5 metric anomaly results are not available in the current session."
else:
    lines = []
    lines.append("Node-5 metric anomaly summary (window 2022-03-20 22:00–22:29 CST):")
    lines.append(f"- Earliest anomaly observed: {earliest}")
    lines.append(f"- {len(df)} (cmdb_id,kpi) pairs flagged in the window (shown up to 20).")
    lines.append("- Typical pattern: most pairs have anomaly_count = 2 and very short consecutive runs (1–2 minutes).")
    lines.append("- Notable examples (kpi | p95 | anomalies | longest_run_min | earliest | severity_ratio):")
    for r in df.head(5).itertuples(index=False):
        lines.append(f"  - {r.kpi_name} | p95={r.p95:.6g} | anomalies={int(r.anomaly_count)} | longest_run_min={int(r.longest_consecutive_anomaly_run_minutes)} | earliest={r.earliest_anomaly_timestamp} | severity_ratio={r.severity_ratio:.6g}")
    lines.append("")
    lines.append("Interpretation: node-5 experienced multiple short, low-to-moderate spikes across memory, disk, network, CPU, and I/O metrics concentrated near the start and through the window (not a prolonged outage). Some metrics (e.g., system.cpu.user, tcp in_segs) show larger relative spikes (severity_ratio > 1), but overall runs are short (≤2 minutes).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                      kpi_name           p95  total_points_in_window  anomaly_count  longest_consecutive_anomaly_run_minutes earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   node-5               system.mem.free  1.581295e+04                      30              2                                        2    2022-03-20 22:00:00 CST         1.583200e+04        0.001205
1   node-5              system.disk.free  5.493571e+08                      30              2                                        2    2022-03-20 22:00:00 CST         5.493604e+08        0.000006
2   node-5         system.net.bytes_sent  3.205826e+04                      30              2                                        1    2022-03-20 22:02:00 CST         3.218109e+04        0.003831
3   node-5   system.net.tcp.retrans_segs  2.000000e-02                      30              1                                        1    2022-03-20 22:03:00 CST         3.000000e-02        0.500000
4   node-5             system.mem.usable  2.402650e+04                       6              1                                        1    2022-03-20 22:07:00 CST         2.402700e+04        0.000021
5   node-5               system.io.svctm  7.855000e-01                      30              2                                        1    2022-03-20 22:11:00 CST         1.500000e+00        0.909612
6   node-5   system.net.udp.in_datagrams  4.781950e+01                      30              2                                        1    2022-03-20 22:18:00 CST         4.962000e+01        0.037652
7   node-5  system.net.udp.out_datagrams  4.781950e+01                      30              2                                        1    2022-03-20 22:18:00 CST         4.962000e+01        0.037652
8   node-5               system.cpu.user  4.070000e+00                      30              2                                        1    2022-03-20 22:19:00 CST         9.170000e+00        1.253071
9   node-5        system.net.tcp.in_segs  1.897040e+02                      30              2                                        1    2022-03-20 22:19:00 CST         4.221000e+02        1.225045
10  node-5       system.net.tcp.out_segs  2.218475e+02                      30              2                                        1    2022-03-20 22:19:00 CST         3.876900e+02        0.747552
11  node-5         system.net.bytes_rcvd  2.320899e+04                      30              2                                        2    2022-03-20 22:19:00 CST         3.806891e+04        0.640266
12  node-5   system.net.packets_in.count  2.153200e+01                      30              2                                        2    2022-03-20 22:19:00 CST         3.190000e+01        0.481516
13  node-5                system.io.util  4.184250e+01                      30              2                                        1    2022-03-20 22:20:00 CST         4.410000e+01        0.053952
14  node-5            system.io.avg_q_sz  4.038100e+01                      30              2                                        1    2022-03-20 22:20:00 CST         4.253000e+01        0.053218
15  node-5  system.net.packets_out.count  2.780200e+01                      30              2                                        1    2022-03-20 22:20:00 CST         2.794000e+01        0.004964
16  node-5     system.os.nofile.used_pct  2.175500e-01                      30              2                                        1    2022-03-20 22:20:00 CST         2.180000e-01        0.002068
17  node-5      system.os.nofile.current  1.425760e+04                      30              2                                        1    2022-03-20 22:20:00 CST         1.427200e+04        0.001010
18  node-5               system.io.await  7.444850e+01                      30              2                                        1    2022-03-20 22:21:00 CST         7.974000e+01        0.071076
19  node-5             system.io.w_await  7.444850e+01                      30              2                                        1    2022-03-20 22:21:00 CST         7.974000e+01        0.071076, '2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 20, 22, 29, 0))
start_ts = int(start_dt.astimezone(pytz.utc).timestamp())
end_ts   = int(end_dt.astimezone(pytz.utc).timestamp())

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)

# KPIs of interest for node-5
kpis = [
    "system.net.tcp.retrans_segs",
    "system.net.tcp.in_segs",
    "system.net.tcp.out_segs",
    "system.net.bytes_rcvd",
    "system.net.bytes_sent"
]

# 1) Compute global P95 and P99 for node-5 & selected KPIs using full series (thresholds BEFORE filtering)
node5_full = metric_df[(metric_df['cmdb_id'] == 'node-5') & (metric_df['kpi_name'].isin(kpis))].copy()
thresholds = (
    node5_full
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p99=lambda s: float(s.quantile(0.99)))
    .reset_index()
)

# 2) Filter metric.csv to incident window and node-5
node5_win = metric_df[
    (metric_df['cmdb_id'] == 'node-5') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['kpi_name'].isin(kpis))
].copy()

# 3) Join with thresholds and compute required metrics per KPI
merged = node5_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
merged = merged.dropna(subset=['p95','p99'])  # drop if no thresholds available

merged['anom_p95'] = merged['value'] > merged['p95']
merged['anom_p99'] = merged['value'] > merged['p99']

def format_ts(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

rows = []
for (cmdb, kpi), g in merged.groupby(['cmdb_id','kpi_name'], sort=False):
    total_points = int(g['value'].count())
    max_val = float(g['value'].max())
    p95_val = float(g['p95'].iloc[0])
    p99_val = float(g['p99'].iloc[0])
    anom_p95_count = int(g['anom_p95'].sum())
    anom_p99_count = int(g['anom_p99'].sum())
    if anom_p95_count > 0:
        earliest_p95_ts = int(g.loc[g['anom_p95'],'timestamp'].min())
        earliest_p95 = format_ts(earliest_p95_ts)
    else:
        earliest_p95 = pd.NA
    if anom_p99_count > 0:
        earliest_p99_ts = int(g.loc[g['anom_p99'],'timestamp'].min())
        earliest_p99 = format_ts(earliest_p99_ts)
    else:
        earliest_p99 = pd.NA
    severity_ratio_p95 = float((max_val - p95_val) / p95_val) if p95_val != 0 else float('nan')
    rows.append({
        'kpi_name': kpi,
        'p95': round(p95_val, 6),
        'p99': round(p99_val, 6),
        'total_points_in_window': total_points,
        'anomaly_count_p95': anom_p95_count,
        'anomaly_count_p99': anom_p99_count,
        'earliest_anomaly_p95': earliest_p95,
        'earliest_anomaly_p99': earliest_p99,
        'max_value_in_window': round(max_val, 6),
        'severity_ratio_p95': severity_ratio_p95
    })

result_df = pd.DataFrame(rows)

# 4) Sort by earliest_anomaly_p95 ascending (NULLs last)
def sort_key(x):
    if pd.isna(x):
        return pd.Timestamp.max
    return pd.to_datetime(x)
result_df['_sort_ts'] = result_df['earliest_anomaly_p95'].apply(lambda x: sort_key(x))
result_df = result_df.sort_values(['_sort_ts'], ascending=[True]).drop(columns=['_sort_ts']).reset_index(drop=True)

# Keep compact and limit to 20 rows
node5_network_table = result_df[[
    'kpi_name','p95','p99','total_points_in_window','anomaly_count_p95','anomaly_count_p99',
    'earliest_anomaly_p95','earliest_anomaly_p99','max_value_in_window','severity_ratio_p95'
]].head(20)

# 5) earliest_node5_network_anomaly scalar
valid_earlies = node5_network_table['earliest_anomaly_p95'].dropna()
earliest_node5_network_anomaly = valid_earlies.min() if not valid_earlies.empty else pd.NA

# Final compact outputs
node5_network_table, earliest_node5_network_anomaly

```
Out[12]:
```
```python
# Build a concise plain-English summary from existing results (node5_network_table, earliest_node5_network_anomaly).
# The summary is a single string variable displayed as output.

try:
    tbl = node5_network_table.copy()
    earliest = earliest_node5_network_anomaly
except NameError:
    summary = "Node-5 network KPI results are not available in the current session."
else:
    lines = []
    lines.append("Node-5 network KPI anomaly summary (window 2022-03-20 22:00–22:29 CST):")
    lines.append(f"- Earliest network anomaly observed: {earliest}")
    lines.append("")
    lines.append("Per-KPI highlights (kpi_name | p95 | p99 | anomalies_p95,p99 | earliest_p95 | max | severity_vs_p95):")
    for r in tbl.itertuples(index=False):
        lines.append(
            f"- {r.kpi_name} | p95={r.p95} | p99={r.p99} | "
            f"anom_p95={int(r.anomaly_count_p95)}, anom_p99={int(r.anomaly_count_p99)} | "
            f"earliest_p95={r.earliest_anomaly_p95} | max={r.max_value_in_window} | "
            f"severity_ratio_p95={r.severity_ratio_p95:.6g}"
        )
    lines.append("")
    lines.append("Interpretation:")
    lines.append("- system.net.bytes_sent and system.net.bytes_rcvd each had 2 p95 exceedances; bytes_sent exceedance at 22:02 CST is small relative to p95 (severity ~0.004).")
    lines.append("- TCP counters (in_segs/out_segs) show 2 p95 exceedances (peak values substantially above p95; in_segs severity >1), indicating notable traffic spikes around 22:19 CST.")
    lines.append("- tcp.retrans_segs exceeded p95 once at 22:03 CST (small absolute value but 50% above p95 — potential packet retransmission spike).")
    lines.append("- Overall: brief but meaningful network activity spikes occurred (traffic and retransmissions) rather than prolonged degradation; earliest event was bytes_sent at 22:02 CST.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      kpi_name         p95         p99  total_points_in_window  anomaly_count_p95  anomaly_count_p99     earliest_anomaly_p95     earliest_anomaly_p99  max_value_in_window  severity_ratio_p95
0        system.net.bytes_sent  32058.2625  32162.8461                      30                  2                  1  2022-03-20 22:02:00 CST  2022-03-20 22:02:00 CST             32181.09            0.003831
1  system.net.tcp.retrans_segs      0.0200      0.0271                      30                  1                  1  2022-03-20 22:03:00 CST  2022-03-20 22:03:00 CST                 0.03            0.500000
2        system.net.bytes_rcvd  23208.9855  34065.2425                      30                  2                  1  2022-03-20 22:19:00 CST  2022-03-20 22:20:00 CST             38068.91            0.640266
3       system.net.tcp.in_segs    189.7040    360.4837                      30                  2                  1  2022-03-20 22:19:00 CST  2022-03-20 22:19:00 CST               422.10            1.225045
4      system.net.tcp.out_segs    221.8475    344.3524                      30                  2                  1  2022-03-20 22:19:00 CST  2022-03-20 22:19:00 CST               387.69            0.747552, '2022-03-20 22:02:00 CST')```
```